In [41]:

from dotenv import load_dotenv

from langchain_core.messages import BaseMessage
from langchain_core.tools import tool

from langchain_google_genai import ChatGoogleGenerativeAI

from langchain_community.tools import DuckDuckGoSearchRun

from langgraph.graph import StateGraph, START
from langgraph.prebuilt import ToolNode, tools_condition

from typing import Annotated, TypedDict
from langgraph.graph.message import add_messages


In [42]:
load_dotenv()

llm = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash"
)

In [43]:
@tool
def add(a:int, b:int) -> int:
    """ Add Two Numbers"""
    return a+b

In [44]:
search = DuckDuckGoSearchRun();

In [45]:
from datetime import datetime


@tool
def current_time() -> str:
    """Get the current date and time."""
    return datetime.now().strftime("%d-%m-%Y %H:%M:%S")

In [46]:
tools = [
    add,
    current_time,
    search
]

In [47]:
llm_with_tools = llm.bind_tools(tools)

In [48]:

class State(TypedDict):
    messages: Annotated[list[BaseMessage], add_messages]

In [49]:
def chatbot(state: State):

    response = llm_with_tools.invoke(
        state["messages"]
    )

    return {
        "messages": [response]
    }

In [50]:
tool_node = ToolNode(tools)

In [51]:
graph = StateGraph(State)
graph.add_node("chatbot", chatbot)

graph.add_node("tools", tool_node)

In [52]:
graph.add_edge(
    START,
    "chatbot"
)

graph.add_conditional_edges(
    "chatbot",
    tools_condition
)

graph.add_edge(
    "tools",
    "chatbot"
)

In [53]:
app = graph.compile()

In [54]:
result = app.invoke({
    "messages": [
        (
            "user",
            "Search the web for the latest information about LangGraph and also calculate 25 + 75."
        )
    ]
})

In [55]:
print(result)

{'messages': [HumanMessage(content='Search the web for the latest information about LangGraph and also calculate 25 + 75.', additional_kwargs={}, response_metadata={}, id='d57e9bbb-792b-4f4d-a505-d543224ec9bc'), AIMessage(content='', additional_kwargs={'function_call': {'name': 'add', 'arguments': '{"a": 25, "b": 75}'}, '__gemini_function_call_thought_signatures__': {'d8e4e45f-dd8b-4590-ae98-4b71fd65a1a8': 'CogCAWkUfRMzmz65Jgn2oE6rIQDLDj6o7wx7FyqG05KPWStyIus7tDRq421aNoPiQITrjTJlt928xD68pTTid+EUYAxALL7ZaqUIHue3sGg7GjlVB8kmCOzA0BMRNOQ/tTG7R81xhAwCCRhv9xS+XR00ZsI3X4b1u2qgg/oDNYLBttobFQI4yNaN69D3LKCX7Ql3baMJK2axWxgpPaDWLfSZ5GzyHwYnEf6Dgpjj9tk/5CgYz5TG4w15b+xwzhtp9kTbaf/ICLBzOUzdf6NvjqauUk/n36nSPpETCxo1VxE4wGTTN/pp5FQ80j9KNlbvKsJp6Vv4yaYE3TS/2tjgBWCWEjx70TXBvgwZ'}}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-2.5-flash', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--01a0cf80-eb37-70a2-9bf2-6d4ceaa1bc8c-0', tool_calls=[{'name': 'duckduckgo_search'